# MCP 서버 사용하기

https://docs.langchain.com/oss/python/langchain/mcp

- AI 애플리케이션이 외부 데이터·기능에 접근하기 위한 표준 프로토콜.
- 도구(함수), 프롬프트 템플릿, 리소스(데이터)를 제공하고, **클라이언트**가 프로토콜에 따라 연결해 사용
- MCP 서버 제공:
  - **Tools**: 호출 가능한 함수 (예: 계산, 날씨 조회, 인사말 생성)
  - **Prompts**: 미리 설계된 프롬프트 템플릿 
  - **Resources**: URI로 구분되는 읽기 전용 데이터 (JSON, 텍스트 등)


In [1]:
from dotenv import load_dotenv
import os

load_dotenv("../../env", override=True)
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
END_POINT = os.getenv("END_POINT")
MODEL_NAME = os.getenv("MODEL_NAME")

os.environ["AZURE_OPENAI_API_KEY"] = AZURE_OPENAI_API_KEY or ""
os.environ["END_POINT"] = END_POINT or ""
os.environ["MODEL_NAME"] = MODEL_NAME or ""

# LangSmith 추적 (선택)
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGCHAIN_ENDPOINT", "")
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_PROJECT"] = "AGENT"

if os.getenv("LANGCHAIN_TRACING_V2") == "true" and os.getenv("LANGCHAIN_API_KEY"):
    print("랭스미스 추적 중:", os.getenv("LANGSMITH_API_KEY", "")[:10])
else:
    print("LangSmith 미사용 또는 키 미설정")
print("MODEL_NAME:", MODEL_NAME)

LangSmith 미사용 또는 키 미설정
MODEL_NAME: gpt-5-nano


In [2]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    model=os.environ["MODEL_NAME"],
    azure_deployment=os.environ["MODEL_NAME"],
    azure_endpoint=os.environ["END_POINT"],
    openai_api_version="2025-03-01-preview",
    openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
)
llm.invoke("안녕!")

AIMessage(content='안녕! 반가워요. 무엇을 도와드릴까요? 대화하고 싶은 주제나 궁금한 게 있으면 말씀해 주세요. 예를 들면:\n- 한국어 공부 팁이나 번역\n- 일반 지식이나 문제 해결\n- 요리 레시피나 여행 정보\n- 코딩이나 수학 문제\n- 편하게 잡담\n\n혹은 지금 바로 하고 싶은 주제가 있나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 421, 'prompt_tokens': 9, 'total_tokens': 430, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DIScxKF4qPrPQM6BEMe0ixnYEihKS', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filter

## 2. MCP 서버 실행


In [ ]:
import subprocess

port = 8000
try:
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)
except Exception:
    pass

MCP_URL = os.getenv("MCP_URL", f"http://localhost:{port}/mcp")

import sys, subprocess
print("using python:", sys.executable)
subprocess.Popen([sys.executable, "mcp_basic_server.py"])

print("MCP 서버 URL:", MCP_URL)

using python: /Users/ktjmh/MyData/GitHub/ktaiagent/.ktaiagent/bin/python
MCP 서버 URL: http://localhost:8000/mcp




╭──────────────────────────────────────────────────────────────────────────────╮
│                                                                              │
│                                                                              │
│                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │
│                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │
│                                                                              │
│                                                                              │
│                                FastMCP 3.1.0                                 │
│                            https://gofastmcp.com                             │
│                                                                              │
│                   🖥  Server:      mcp_basic_example, 3.1.0                   │
│                   🚀 Deploy free: https://fastmcp.cloud                      │
│                          

## 3. 세션 직접 사용

1. 서버와 연결 (여기서는 streamable_http)
2. **ClientSession**으로 JSON-RPC 2.0 세션을 열고 initialize()로 핸드셰이크(버전 확인, 기능 및 구현 정보 공유)
3. load_mcp_tools, load_mcp_prompt, load_mcp_resources로 서버가 제공하는 항목을 가져옵니다.

async with로 세션을 열어두는 동안만 tools/prompts/resources를 사용할 수 있고, 블록을 벗어나면 연결이 종료됩니다.

In [4]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_mcp_adapters.prompts import load_mcp_prompt
from langchain_mcp_adapters.resources import load_mcp_resources

async with streamablehttp_client(MCP_URL) as (read, write, _):
    async with ClientSession(read, write) as session:
        handshake = await session.initialize()
        print("서버 정보:", handshake.serverInfo)
        print("capabilities:", handshake.capabilities)

        tools = await load_mcp_tools(session)
        prompts = await load_mcp_prompt(session, "language", arguments={"country": "Korea", "name": "Jintae"})
        resources = await load_mcp_resources(session, uris="myinfo://app-info")

print("\n--- Tools ---")
print(tools)
print("\n--- Prompt (language) ---")
print(prompts)
print("\n--- Resources ---")
print(resources) 

INFO:     127.0.0.1:57812 - "POST /mcp HTTP/1.1" 200 OK
서버 정보: name='mcp_basic_example' title=None version='3.1.0' websiteUrl=None icons=None
capabilities: experimental={} logging=None prompts=PromptsCapability(listChanged=True) resources=ResourcesCapability(subscribe=False, listChanged=True) tools=ToolsCapability(listChanged=True) completions=None tasks=None extensions={'io.modelcontextprotocol/ui': {}}
INFO:     127.0.0.1:57815 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57816 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57818 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57820 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57822 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57824 - "DELETE /mcp HTTP/1.1" 200 OK

--- Tools ---
[StructuredTool(name='say_hello', description='이름(name)을 받아서 인사말을 반환합니다.', args_schema={'additionalProperties': False, 'properties': {'name': {'type': 'string'}}, 'required': ['name'], 'type': 'object'}, metadata={'_meta': {'fastmcp':

In [5]:
print(tools)

[StructuredTool(name='say_hello', description='이름(name)을 받아서 인사말을 반환합니다.', args_schema={'additionalProperties': False, 'properties': {'name': {'type': 'string'}}, 'required': ['name'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x114c69800>)]


### 3.1 Tools 사용

ainvoke로 인자를 넘겨 호출할 수 있음


In [6]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langchain_mcp_adapters.tools import load_mcp_tools

async with streamablehttp_client(MCP_URL) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await load_mcp_tools(session)
        print('tools:', tools)
        result = await tools[0].ainvoke({"name": "Jintae"})
        print("say_hello 결과:", result)
try:
    result = await tools[0].ainvoke({"name": "Jintae"})
    print("say_hello 결과:", result)
except Exception as e:
    print("에러 발생")

INFO:     127.0.0.1:57922 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57925 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:57926 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:57928 - "POST /mcp HTTP/1.1" 200 OK
tools: [StructuredTool(name='say_hello', description='이름(name)을 받아서 인사말을 반환합니다.', args_schema={'additionalProperties': False, 'properties': {'name': {'type': 'string'}}, 'required': ['name'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x114c699e0>)]
INFO:     127.0.0.1:57930 - "POST /mcp HTTP/1.1" 200 OK
say_hello 결과: [{'type': 'text', 'text': 'Hello, Jintae! Nice to meet you.', 'id': 'lc_09713c56-33e5-405c-b859-0dd2ae8fcc35'}]
INFO:     127.0.0.1:57932 - "DELETE /mcp HTTP/1.1" 200 OK
에러 발생


### 3.2 Prompts 사용

미리 설계된 프롬프트 템플릿 

**HumanMessage** 형태로 반환되므로, 그대로 LLM에 넘겨 사용할 수 있습니다.

In [7]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langchain_mcp_adapters.prompts import load_mcp_prompt

async with streamablehttp_client(MCP_URL) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        prompts = await load_mcp_prompt(session, "language", arguments={"country": "Korea", "name": "Jintae"})
        print('prompts:', prompts)
        reply = llm.invoke(prompts)
        print("LLM :", reply)

INFO:     127.0.0.1:58050 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58053 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58054 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58056 - "POST /mcp HTTP/1.1" 200 OK
prompts: [HumanMessage(content="해당 나라의 언어로 짧고 친절하게 인사해줘. 국가 : 'Korea', 이름 : 'Jintae'", additional_kwargs={}, response_metadata={})]
LLM : content='안녕하세요, 진태님! 반갑습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 533, 'prompt_tokens': 37, 'total_tokens': 570, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DIStbxGGoWwM8dLUoRNqYOc9ya2xm', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filt

### 3.3 Resources 사용

URI로 구분되는 읽기 전용 데이터

서버에서는 **Blob**(Binary Large Object) 형태로 반환하며, 텍스트·JSON·이미지(base64) 등 가능. 

Blob : 이미지, 텍스트, pdf 등 모든 데이터를 이진화로 표현된 변경할 수 없는 로우 데이터

In [8]:
import ast
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langchain_mcp_adapters.resources import load_mcp_resources

async with streamablehttp_client(MCP_URL) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        resources = await load_mcp_resources(session, uris="myinfo://app-info")
        print('resources:', resources)
        print('resources[0]:', resources[0])
        for r in resources:
            print("uri:", r.metadata["uri"])
            data = ast.literal_eval(r.data)
            for k, v in data.items():
                print(f"  {k}: {v}")

INFO:     127.0.0.1:58340 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58343 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58344 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58346 - "POST /mcp HTTP/1.1" 200 OK
resources: [Blob 4643758848]
resources[0]: metadata={'uri': 'myinfo://app-info'} data='{"name": "mcp_basic_example", "description": "앱의 구성요소를 제공합니다.", "version": "0.0.1", "author": "Kim jintae", "email": "jt.kim@fake.com"}' mimetype='text/plain'
uri: myinfo://app-info
  name: mcp_basic_example
  description: 앱의 구성요소를 제공합니다.
  version: 0.0.1
  author: Kim jintae
  email: jt.kim@fake.com
INFO:     127.0.0.1:58348 - "DELETE /mcp HTTP/1.1" 200 OK


---
## 4. MultiServerMCPClient

MultiServerMCPClient는 여러 MCP 서버를 한 번에 등록하고, 필요할 때만 세션을 열어 get_tools, get_prompt, get_resources를 호출하는 방식

세션을 직접 관리하지 않아도 되고, 호출 시마다 내부에서 연결을 열고 닫습니다.

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "server_basic": {
            "transport": "streamable_http",
            "url": MCP_URL,
        }
    }
)

tools = await client.get_tools()
prompt_msg = await client.get_prompt("server_basic", "language", arguments={"country": "Korea", "name": "Jintae"})
resources = await client.get_resources("server_basic", uris="myinfo://app-info")
print('>> call get_tools() ================')
print("Tools:", await tools[0].ainvoke({"name": "Jintae"}))


INFO:     127.0.0.1:58353 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58356 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58357 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58359 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58361 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58363 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58366 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58367 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58369 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58371 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58373 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58376 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58377 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58379 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58381 - "DELETE /mcp HTTP/1.1" 200 OK
>> call get_tools() ================
INFO:     127.0.0.1:58383 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58386 - "POST /mcp HTTP/1.

## 5. 에이전트에 MCP 툴 연결

MCP에서 가져온 tools를 에이전트에 그대로 바인딩할 수 있습니다. 


In [10]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

tools = await client.get_tools()

PROMPT = """
당신은 친절한 안내원입니다. 유저에게 친절하게 인사하세요.
도구를 적절히 사용해 유저의 요청을 처리하세요.
상대가 이름을 밝힐 경우 say_hello 툴을 사용하세요.
상대가 인사를 원하지만 이름을 밝히지 않을 경우 임의의 이름을 만들어 say_hello 툴을 사용하세요.
tool을 사용할 필요가 없는 일반적인 질문에는 tool을 호출하지 말고 간단하고 친절하게 답변하세요.
"""

agent = create_agent(
    llm,
    tools,
    system_prompt=PROMPT,
    checkpointer=InMemorySaver(),
    debug=False,
)

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "홍길동에게 인사해줘."}]},
    config={"configurable": {"thread_id": "1"}},
)
print(result["messages"][-1].content)

INFO:     127.0.0.1:58397 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58400 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58401 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58403 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58405 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58408 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58411 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58412 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58414 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58416 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58418 - "DELETE /mcp HTTP/1.1" 200 OK
홍길동님께 인사를 전했습니다: Hello, 홍길동! Nice to meet you.

다른 분께도 인사를 원하시면 이름을 알려주세요.


### 5.1 리소스를 툴로 감싸서 에이전트에 추가

MCP **Resources**는 에이전트가 직접 호출하는 툴이 아니므로, "앱 정보를 알려줘" 같은 요청을 처리하려면 **리소스를 읽어 반환하는 툴**을 하나 만들어 에이전트에 추가할 수 있습니다.

In [11]:
from langchain.tools import tool

@tool
async def get_app_info(x: str = "") -> str:
    """MCP 서버에서 제공하는 앱의 정보를 반환합니다. 입력으로는 아무 문자열이나 넣습니다."""
    print("get_app_info 호출")
    blobs = await client.get_resources("server_basic", uris="myinfo://app-info")
    return blobs[0].data if blobs else ""

tools_with_info = await client.get_tools() + [get_app_info]

PROMPT_EXT = """
당신은 친절한 안내원입니다. 유저에게 친절하게 인사하세요.
도구를 적절히 사용해 유저의 요청을 처리하세요.
상대가 이름을 밝힐 경우 say_hello 툴을 사용하세요.
앱의 정보는 get_app_info에 들어 있습니다. 사용자가 앱 정보를 물어보면 get_app_info를 사용하세요.
tool이 필요 없는 일반적인 질문에는 tool을 호출하지 말고 간단하고 친절하게 답변하세요.
"""

agent_ext = create_agent(
    llm,
    tools_with_info,
    system_prompt=PROMPT_EXT,
    checkpointer=InMemorySaver(),
    debug=False,
)

result = await agent_ext.ainvoke(
    {"messages": [{"role": "user", "content": "앱의 이름이 뭐야?"}]},
    config={"configurable": {"thread_id": "2"}},
)
print(result["messages"][-1].content)

INFO:     127.0.0.1:58424 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58427 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58428 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58430 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58432 - "DELETE /mcp HTTP/1.1" 200 OK
get_app_info 호출
INFO:     127.0.0.1:58434 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58437 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58438 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58440 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58442 - "DELETE /mcp HTTP/1.1" 200 OK
안녕하세요! 앱의 이름은 mcp_basic_example입니다. 필요하시면 버전이나 설명도 같이 알려드릴게요.


In [12]:
result = await agent_ext.ainvoke(
    {"messages": [{"role": "user", "content": "이 앱의 제작자에 대한 정보를 알려줘"}]},
    config={"configurable": {"thread_id": "2"}},
)
print(result["messages"][-1].content)

get_app_info 호출
INFO:     127.0.0.1:58447 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58450 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:58451 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58453 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:58455 - "DELETE /mcp HTTP/1.1" 200 OK
안녕하세요! 이 앱의 제작자 정보는 아래와 같습니다.
- 이름(제작자): Kim jintae
- 이메일: jt.kim@fake.com

다른 정보가 필요하시면 말씀해 주세요.
